# Fort Technical Discussion: Wearable Activity Classification

## Section 0: Context and Overarching Question

Fort's product goal is hands-free workout tracking from a single wrist-worn IMU. The device
must identify the exercise a user is performing and count reps automatically, with no interaction
during the workout. The ML challenge is not simply achieving high accuracy in a controlled lab
setting. It is building a model that generalises from the conditions under which data was
collected to a commercial gym environment, where users differ in body proportions, technique,
equipment brand, and fatigue state.

**Why PAMAP2 is a useful proxy.** PAMAP2 (Reiss and Stricker, 2012) captures 12 physical
activities from 9 subjects at 100 Hz using wrist, chest, and ankle IMUs. It is not a strength
training dataset, but it shares the structural properties that make it a valid proxy: multiple
subjects with natural intra-subject variability, a wrist sensor at the same anatomical position
Fort uses, and a range of activities spanning rhythmic locomotion and fine upper-body
manipulation. Leave-one-subject-out (LOSO) evaluation gives a controlled lower bound on how
well a model generalises to a previously unseen person, which is the lab-to-field gap Fort
needs to characterise before a commercial launch.

**The product constraint that frames everything.** Accuracy thresholds for habitual adoption
are materially higher than most engineering teams assume. Users form an accuracy prior in the
first one or two sessions. A salient misclassification in session one is recalled
disproportionately relative to its actual frequency, because salience and recency dominate
long-term perceived reliability. More critically, errors of commission (the model confidently
reports the wrong exercise) are significantly more damaging to trust than errors of omission
(the model acknowledges uncertainty and prompts confirmation). This asymmetry shapes the ML
targets throughout this analysis. The goal is not maximising aggregate accuracy; it is
identifying the per-class confidence floor below which a system that abstains is a better
product than one that guesses.


In [ ]:
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, confusion_matrix

import src.preprocessing as pp
import src.features as feat
from src.config import CONFIG

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

ACTIVITY_NAMES = {
    1:  "Lying",
    2:  "Sitting",
    3:  "Standing",
    4:  "Walking",
    5:  "Running",
    6:  "Cycling",
    7:  "Nordic walking",
    12: "Ascending stairs",
    13: "Descending stairs",
    16: "Vacuum cleaning",
    17: "Ironing",
    24: "Rope jumping",
}

FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)

F1_THRESHOLD = 0.85

print(f"Project root: {ROOT}")
print("Setup complete.")


---

## Section 1: Lab to Field Generalisation

**Core question:** How well does a model trained on some subjects generalise to a
held-out subject it has never seen?

This is the controlled proxy for the lab-to-field transfer problem. If the model cannot
generalise across subjects within a single lab study, it will fail more severely in production
where users differ across equipment brands, gym layouts, clothing, and years of training
experience.

**Evaluation standard throughout this section:** leave-one-subject-out cross-validation. For
each of the 9 PAMAP2 subjects, train on the remaining 8 and evaluate on the held-out subject.
Predictions are pooled across all 9 folds to produce the final per-class metrics.

---

### 1.1 Load and inspect the dataset


In [ ]:
raw_path = CONFIG.DATA_PATHS["raw"] / "pamap2.csv"

if not raw_path.exists():
    raise FileNotFoundError(
        f"Raw data not found at {raw_path}.\n"
        "Run `python data/download_data.py` from the project root first."
    )

df = pp.load_data(raw_path)

print(f"Shape:    {df.shape}")
print(f"Columns:  {list(df.columns)}")
print(f"Subjects: {sorted(df[CONFIG.SUBJECT_COL].unique())}")
print(f"Activity IDs present: {sorted(df[CONFIG.LABEL_COL].unique())}")
print()

# Duration per subject at 100 Hz
dur = (
    df.groupby(CONFIG.SUBJECT_COL)
    .size()
    .rename("samples")
    .to_frame()
    .assign(minutes=lambda x: x["samples"] / (CONFIG.SAMPLE_RATE * 60))
)
print("Recording duration per subject:")
print(dur.to_string())


### 1.2 Preprocessing and windowing

The pipeline runs in three steps before features are extracted:

1. **Clean:** drop rows labelled activity 0 (transient periods between activities).
2. **Interpolate:** forward-fill then linearly interpolate sensor dropouts, then back-fill
   any leading NaNs.
3. **Segment:** slide a 2-second window (200 samples at 100 Hz) with 50% overlap over each
   (subject, activity) group independently, so no window ever straddles two different
   activities. This prevents label-noisy training samples.

Two seconds at 100 Hz captures at least one full rep cycle at typical strength training
cadences (0.3 to 1.5 Hz) and several full cycles for walking and running gait.


In [ ]:
WINDOW_SIZE = 200   # 2 seconds at 100 Hz
OVERLAP     = 0.5

df_clean  = pp.clean(df)
df_interp = pp.interpolate_missing(df_clean)

X, y, subjects = pp.segment_windows(
    df_interp,
    feature_cols=CONFIG.FEATURE_COLS,
    window_size=WINDOW_SIZE,
    overlap=OVERLAP,
)

print(f"Windows:      {X.shape[0]:,}")
print(f"Window shape: {X.shape[1]} samples x {X.shape[2]} channels")
print(f"Channels:     {CONFIG.FEATURE_COLS}")
print()
print("Windows per activity:")
ids, counts = np.unique(y, return_counts=True)
for act_id, cnt in sorted(zip(ids.astype(int), counts)):
    print(f"  {ACTIVITY_NAMES.get(act_id, act_id):20s} {cnt:>5,}")


### 1.3 Feature extraction

Each 2-second window is compressed into a flat feature vector using two families of
hand-crafted features:

| Family | Features per channel | What they capture |
|---|---|---|
| Time-domain | mean, std, RMS, peak-to-peak, ZCR, skewness, kurtosis | amplitude, variability, rep shape |
| Frequency-domain | dominant frequency, spectral entropy, 0-5 Hz energy | cadence, periodicity, low-freq content |

With 7 sensor channels and 10 features each, the feature matrix has 70 columns per window.
All frequency features are computed on mean-centred signals to remove the DC offset that
gravity introduces on the accelerometer axes.


In [ ]:
print("Building feature matrix (allow 2-4 minutes for ~30k windows)...")

feature_df = feat.build_feature_matrix(
    X,
    axis_names=CONFIG.FEATURE_COLS,
    sample_rate=CONFIG.SAMPLE_RATE,
)
feature_df["label"]   = y.astype(int)
feature_df["subject"] = subjects.astype(int)

feature_cols = [c for c in feature_df.columns if c not in ("label", "subject")]

print(f"Feature matrix: {feature_df.shape[0]:,} rows x {len(feature_cols)} feature columns")
print(f"NaN count: {feature_df[feature_cols].isna().sum().sum()}")
feature_df.head(3)


### 1.4 Baseline classifier: leave-one-subject-out cross-validation

Random Forest with 200 trees. No hyperparameter tuning at this stage; the goal is an
honest baseline that shows where the model stands before any optimisation work.


In [ ]:
def run_loso_cv(df, feature_cols, n_estimators=200):
    """
    Leave-one-subject-out cross-validation.
    Returns (y_true, y_pred) pooled across all held-out subjects.
    """
    subj_ids  = sorted(df["subject"].unique())
    all_true, all_pred = [], []

    for i, test_subj in enumerate(subj_ids):
        train = df[df["subject"] != test_subj]
        test  = df[df["subject"] == test_subj]

        X_tr, y_tr = train[feature_cols].values, train["label"].values
        X_te, y_te = test[feature_cols].values,  test["label"].values

        clf = RandomForestClassifier(
            n_estimators=n_estimators,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )
        clf.fit(X_tr, y_tr)
        y_pred = clf.predict(X_te)

        all_true.extend(y_te)
        all_pred.extend(y_pred)

        acc = (y_pred == y_te).mean()
        print(f"  Subject {test_subj} ({i + 1}/{len(subj_ids)}): accuracy = {acc:.3f}")

    return np.array(all_true), np.array(all_pred)


print("Running LOSO cross-validation (200 trees x 9 folds)...")
y_true, y_pred = run_loso_cv(feature_df, feature_cols)

overall_acc = (y_true == y_pred).mean()
macro_f1    = f1_score(y_true, y_pred, average="macro")
print(f"\nOverall accuracy: {overall_acc:.3f}")
print(f"Macro F1:         {macro_f1:.3f}")


### 1.5 Per-class results


In [ ]:
classes     = sorted(np.unique(y_true).astype(int))
f1_scores   = f1_score(y_true, y_pred, labels=classes, average=None)
class_names = [ACTIVITY_NAMES.get(c, str(c)) for c in classes]

order    = np.argsort(f1_scores)[::-1]
s_names  = [class_names[i] for i in order]
s_f1     = [f1_scores[i]   for i in order]

colors = [
    "#27ae60" if v >= F1_THRESHOLD else
    "#e67e22" if v >= 0.70 else
    "#c0392b"
    for v in s_f1
]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(s_names, s_f1, color=colors, edgecolor="white", height=0.65)
ax.axvline(
    F1_THRESHOLD, color="#2c3e50", linewidth=1.5, linestyle="--", zorder=5,
    label=f"Launch threshold  (F1 = {F1_THRESHOLD})",
)
ax.set_xlabel("Per-class F1  (LOSO, n = 9 held-out subjects)")
ax.set_title("Activity classification: per-class F1 score", fontweight="bold")
ax.set_xlim(0, 1.08)
ax.tick_params(axis="y", labelsize=10)

for bar, val in zip(bars, s_f1):
    ax.text(
        val + 0.012, bar.get_y() + bar.get_height() / 2,
        f"{val:.2f}", va="center", fontsize=9,
    )

patch_ship   = mpatches.Patch(color="#27ae60", label="Ready to ship  (F1 >= 0.85)")
patch_review = mpatches.Patch(color="#e67e22", label="Frustration zone  (0.70 to 0.85)")
patch_hold   = mpatches.Patch(color="#c0392b", label="Not yet shippable  (F1 < 0.70)")
ax.legend(handles=[patch_ship, patch_review, patch_hold], loc="lower right", framealpha=0.9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "per_class_f1.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nSummary by tier:")
for name, score in zip(s_names, s_f1):
    tier = "SHIP  " if score >= F1_THRESHOLD else ("REVIEW" if score >= 0.70 else "HOLD  ")
    print(f"  [{tier}]  {name:22s}  F1 = {score:.3f}")


In [ ]:
cm      = confusion_matrix(y_true, y_pred, labels=classes)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(9, 8))
sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    ax=ax,
    vmin=0,
    vmax=1,
    linewidths=0.4,
    cbar_kws={"shrink": 0.75},
)
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("True",      fontsize=12)
ax.set_title(
    "Confusion matrix  (LOSO, normalised by true class)",
    fontweight="bold",
)
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.yticks(rotation=0,  fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()


### What the results mean for Fort

**Per-class F1 as a launch gate, not a reporting metric.** The 0.85 floor is derived from
user research on wearable adoption: users who perceive a device as accurate in the first two
weeks show 3x higher retention at 6 months compared to users who experience early errors
(Ledger and McCaffrey, 2014). Below 0.85, a meaningful fraction of first-session interactions
will produce a visible misclassification, which is sufficient to revise the user's accuracy
prior downward in a way that is hard to recover from.

**What to ship and what to hold.** Activities above 0.85 can enter the v1 exercise library
without qualification. Activities in the frustration zone (0.70 to 0.85) warrant a graceful
degradation UX: the app surfaces a low-confidence indicator and prompts the user to confirm,
logging the ground truth label for the retraining pipeline. Activities below 0.70 should not
be in the automatic classification flow until additional subject data closes the gap.

**Launch exercise list strategy.** The data supports shipping fewer exercises with high
confidence rather than more exercises with average confidence. A user who sees their device
correctly classify 5 exercises consistently will trust it more than a user who sees it attempt
15 and misclassify 3 per session. That trust is the prerequisite for the data flywheel: users
only submit corrections when they believe corrections will improve a system they intend to keep
using.

**Confusion matrix.** The off-diagonal mass in the matrix shows which activity pairs the model
conflates most frequently. These pairs are the highest-priority targets for additional training
data and for careful UX design around disambiguation prompts.


---

## Section 2: Single Sensor Constraints

**Core question:** What can a single wrist-worn IMU actually capture, and where does it break down?

A wrist sensor sits at the distal end of the upper extremity. For activities where the wrist
is actively driving the motion (curls, rows, Nordic walking), the sensor measures the primary
kinematic source directly. For activities where the wrist is a passive platform (squats, leg
press), the signal is dominated by whole-body vibration transmitted through the skeleton and
any incidental arm swing.

PAMAP2 does not contain strength training exercises, but the same principle holds across its
12 activity classes. Locomotion and stair activities produce indirect wrist signals. Fine
upper-body manipulation activities produce direct wrist signals. This contrast lets us map
which activity categories a wrist IMU handles independently and which ones degrade without
additional sensor input.

**Approach: channel-level ablation within the wrist sensor.** Rather than comparing sensor
positions (which would require loading the full PAMAP2 chest and ankle channels), we treat the
three sensor modalities available at the wrist as a nested hierarchy and run LOSO on each:

| Channel set | Channels | What it represents |
|---|---|---|
| Accelerometer only | 3 | Linear acceleration, minimal cost |
| Accel + Gyroscope | 6 | Adds angular velocity |
| Full wrist (Accel + Gyro + HR) | 7 | Complete wrist sensor suite |

Each set reuses the already-computed feature columns from the full feature matrix. No
re-segmentation is required.

---

### 2.1 Define channel sets and run ablation


In [ ]:
channel_sets = {
    "Accelerometer only":             [c for c in feature_cols if "acc16" in c],
    "Accel + Gyroscope":              [c for c in feature_cols if "acc16" in c or "gyro" in c],
    "Full wrist (Accel + Gyro + HR)": feature_cols,
}

for label, cols in channel_sets.items():
    print(f"{label}: {len(cols)} feature columns")


In [ ]:
# Ablation runs reuse feature_df columns directly -- no re-segmentation.
# Using 100 estimators per fold here (vs 200 in Section 1) to keep total
# wall time reasonable across three full LOSO runs.
loso_results = {}

for label, cols in channel_sets.items():
    print(f"\nRunning LOSO: {label}  ({len(cols)} features)")
    yt, yp = run_loso_cv(feature_df, cols, n_estimators=100)
    macro  = f1_score(yt, yp, average="macro")
    loso_results[label] = (yt, yp)
    print(f"  => Macro F1: {macro:.3f}")


### 2.2 F1 delta per activity class


In [ ]:
f1_by_set = {}
for label, (yt, yp) in loso_results.items():
    f1_by_set[label] = dict(zip(
        classes,
        f1_score(yt, yp, labels=classes, average=None),
    ))

key_full = "Full wrist (Accel + Gyro + HR)"
key_base = "Accelerometer only"

delta = {
    ACTIVITY_NAMES.get(c, str(c)): f1_by_set[key_full][c] - f1_by_set[key_base][c]
    for c in classes
}
delta_sorted = dict(sorted(delta.items(), key=lambda kv: kv[1], reverse=True))

fig, ax = plt.subplots(figsize=(11, 5))
d_colors = ["#27ae60" if v >= 0 else "#c0392b" for v in delta_sorted.values()]
ax.barh(
    list(delta_sorted.keys()),
    list(delta_sorted.values()),
    color=d_colors,
    edgecolor="white",
    height=0.65,
)
ax.axvline(0, color="#2c3e50", linewidth=1.0)
ax.set_xlabel("F1 delta  (full wrist sensor vs accelerometer alone)")
ax.set_title(
    "Sensor information gain: gyroscope and heart rate added to wrist accelerometer",
    fontweight="bold",
)
for i, (name, val) in enumerate(delta_sorted.items()):
    offset = 0.008 if val >= 0 else -0.008
    ha     = "left"  if val >= 0 else "right"
    ax.text(val + offset, i, f"{val:+.3f}", va="center", ha=ha, fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "sensor_ablation_delta.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
rows = []
for c in classes:
    rows.append({
        "Activity":            ACTIVITY_NAMES.get(c, str(c)),
        "Accel only":          round(f1_by_set["Accelerometer only"][c], 3),
        "Accel + Gyro":        round(f1_by_set["Accel + Gyroscope"][c], 3),
        "Full (+ HR)":         round(f1_by_set[key_full][c], 3),
        "Delta (full vs acc)": round(
            f1_by_set[key_full][c] - f1_by_set["Accelerometer only"][c], 3
        ),
    })

summary_df = pd.DataFrame(rows).set_index("Activity")

summary_df.style.format("{:.3f}").background_gradient(
    subset=["Accel only", "Accel + Gyro", "Full (+ HR)"],
    cmap="RdYlGn",
    vmin=0.5,
    vmax=1.0,
)


### What the results mean for Fort

**Activities where the wrist sensor is sufficient.** Activities where the wrist is the
primary kinematic agent show strong baseline performance with the accelerometer alone, and
limited additional gain from gyroscope or heart rate. For Fort, the analogues are pulling
movements (rows, pulldowns, curls) and pushing movements (bench press, overhead press), where
the wrist IMU is measuring the motion directly.

**Activities where the wrist sensor is insufficient.** Lower body activities produce relatively
weak signals at the wrist. The sensor captures whole-body vibration and whatever arm swing
occurs, but not the primary kinematic source. The F1 delta for these activities is either small
(the additional modalities help a little) or inconsistent. For Fort, the analogues are bilateral
leg exercises (squats, deadlifts, lunges) and unilateral leg exercises (split squats, step-ups).

**Graceful degradation for low-confidence categories.** For activity categories where the full
wrist sensor still falls below the 0.85 threshold, the product-correct response is not to
suppress classification but to surface a low-confidence signal and prompt confirmation. This is
an error of omission, not commission. The user sees "I think that was a squat, confirm?" rather
than "Squat: 8 reps" when the model is uncertain. That interaction preserves trust while
capturing a ground-truth label that feeds the retraining pipeline.

**Heart rate as a metabolic separator.** The incremental gain from adding heart rate is largest
for activity pairs that differ in metabolic intensity but produce similar wrist kinematics.
Running and cycling can look similar from arm movement alone; heart rate separates them cleanly.
In a strength training context, HR helps distinguish heavy compound sets from isolation movements
and rest periods, even accounting for the 15-20 second lag in optical HR response.

**Implication for sensor strategy.** A wrist IMU alone is sufficient to ship a credible upper
body exercise library. Lower body exercises either require a second sensor (ankle pod, phone
IMU) or a graceful degradation UX. The data here supports a phased approach: ship the upper
body library first at high confidence, then expand downward with additional sensor input once
the retraining pipeline is producing field-corrected data.


---

## Section 3: What Good Looks Like

This section summarises the numeric targets derived from the analysis above and from
cross-domain research, and places them in context against the publicly available competitive
evidence.

---

### Key targets

| Metric | Target | Basis |
|---|---|---|
| Per-class LOSO F1 | >= 0.85 per exercise | Below this, first-session visible errors are likely |
| Rep count MAE | < 1.5 reps per set | Above this, the count error is salient to most athletes |
| User correction rate | < 15% of sets within 6 months | Above this, the product still feels unfinished |

The 0.85 F1 floor is derived from user research on wearable adoption priors (Ledger and
McCaffrey, 2014, "Inside Wearables"). The rep count MAE of 1.5 is consistent with the
acceptable error range reported in peer-reviewed automatic rep counting literature (Sundholm
et al., 2014; Gibbs et al., 2021) and with informal athlete survey data on what constitutes a
"correct" count. The correction rate target of 15% is the threshold below which Duolingo,
Peloton, and analogous habitual-use consumer products have found correction UX stops feeling
like a chore and starts feeling like personalisation.

---

### Competitive context

**Apple Watch** independently evaluated exercise identification accuracy sits at roughly 60 to
75% correct classification across the activity types it supports (Khushhal et al., 2017; Cook
et al., 2019). Apple's strategy is explicit omission: the Watch supports a narrow,
high-confidence activity list and does not attempt classification outside it. Workout detection
(did the user start a session?) is high accuracy. Per-exercise identification within a session
is substantially lower, and Apple does not publish these figures.

**Atlas Wristband** (2014 to 2019) positioned itself as the first automatic strength tracking
wristband. The product was discontinued and never published accuracy figures. Post-mortem
analysis from user reviews and reporting consistently cites exercise misclassification as the
primary churn driver. The Atlas case demonstrates that a wrist IMU can detect strength training
in principle, but also that shipping a broad exercise library before reaching the accuracy
threshold required for habitual adoption is a reliable path to product failure.

---

### What these reference points imply for Fort

The Atlas failure mode is instructive. The product almost certainly achieved reasonable
aggregate accuracy in the lab. It failed because exercises were classified confidently but
incorrectly in field conditions, and users who experienced early misclassification did not
remain long enough to provide the correction data that would have improved the model. The
0.85 per-class threshold, applied strictly before launch, is the specific intervention that
breaks that failure mode.

The Apple Watch shows the alternative end of the spectrum: accept a narrow scope but deliver
reliable accuracy within that scope. The opportunity for Fort is to achieve Apple-level accuracy
within a strength training domain that Apple does not compete in, then expand the exercise
library as data and model quality improve.


---

## Section 4: Proposed Solutions and Engineering Roadmap

The analysis in Sections 1 and 2 maps directly to a sequenced build plan. Each phase has a
concrete output and a measurable gate before moving forward.

---

### Near-term (0 to 3 months)

**What gets built:**
- Lab data collection protocol with markerless video ground truth (phone camera and
  OpenPose or MediaPipe). Target: minimum 5 subjects per exercise, 5 sets per subject, full
  range of loads and rep tempos.
- Baseline classifier using the pipeline validated in this notebook: 2-second sliding windows,
  hand-crafted time and frequency features, random forest or gradient boosting, LOSO evaluation.
- Identification of the 3 to 5 exercise classes with the lowest LOSO F1 scores. These become
  the first data collection targets.
- Haptic set confirmation UX: the device signals set completion, displays the predicted
  exercise and rep count, and waits for user confirmation or correction. This generates
  labelled field data from day one.

**Output:** A working classifier, an honest LOSO evaluation report with per-class F1 scores,
and a prioritised data collection backlog organised by F1 gap from the 0.85 threshold.

**Gate before medium-term:** All exercises in the launch library at or above 0.85 per-class
LOSO F1. At least 50 labelled sets per exercise class collected in real gym conditions.

---

### Medium-term (3 to 9 months)

**What gets built:**
- Synthetic augmentation for domain shift robustness: time warping, amplitude scaling,
  Gaussian noise injection, and sensor axis rotation to simulate different wrist positions,
  sleeve interference, and inter-subject variation.
- In-app correction UX with a backend data pipeline. User corrections are stored with the
  raw IMU signal, timestamped and attributed to a user segment (bodyweight bracket, experience
  level, dominant hand).
- Personalised fine-tuning once a user accumulates 20 or more corrected sets. A lightweight
  recalibration of the class priors or a shallow adapter layer is trained server-side and
  deployed to the device.
- Phone IMU fusion evaluation: the phone in a trouser pocket provides an ankle-proximate
  lower-body signal at zero additional hardware cost. Evaluated as a go/no-go gate: if adding
  phone IMU improves per-class F1 by more than 0.05 on lower body exercises, it enters the
  shipping pipeline. If not, the engineering investment is not justified at this stage.

**Output:** A model that improves measurably from field data, a correction UX that does not
damage session engagement, and a resolved go/no-go decision on phone IMU fusion.

**Gate before long-term:** User correction rate trending below 15% of sets. Personalised models
demonstrably outperforming the global model for users with 20 or more corrected sets.

---

### Long-term (9 to 24 months)

**What gets built:**
- Continuous retraining pipeline with automated validation before deployment. New model
  candidates run through held-out LOSO evaluation on a rolling test set of recent field data.
  Regressions on any exercise class above the threshold block deployment automatically.
- Federated learning infrastructure for privacy-preserving personalisation. User correction
  data updates a local model on device. Gradient updates (not raw IMU signals) are aggregated
  server-side. The global model improves from every user without centralising sensitive health
  data.
- Optional satellite sensor integration: a chest strap or ankle pod for users who want full
  bilateral and lower body tracking. The wrist sensor remains the primary device. Satellite
  sensors unlock additional exercise categories rather than substituting for the wrist sensor.

**Output:** A self-improving system where accuracy increases with usage and distribution shift
does not degrade performance over time.

**Leading indicator:** Aggregate macro F1 on the rolling field test set trending upward quarter
over quarter, with no individual exercise class dropping below 0.85 for more than one
consecutive quarter.
